In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src import setup_workspace, query_to_df, ds3on1

# 1. Inicializar workspace y rutas
paths = setup_workspace()

# 2. Consultar directamente a PostgreSQL (vía DuckDB ATTACH)
#df = query_to_df("SELECT * FROM db.EuInfo;")

# 3. Análisis univariable 3-en-1
#ds3on1(df, 'happy', Desc=True, Stat=True, Graph=True)

In [ ]:
import pandas as pd
import numpy as np
import duckdb as db
#Lets prepare the country code dictionary to ensure our economical dataframe, has all the data for every country we have from surveys.
df_raw = query_to_df("""
    SELECT emplno, agea, emplrel, edulvlb, happy, lrscale, cntry, EmployType
    FROM db.surveys
    WHERE COLUMNS(*) IS NOT NULL
""")

df_62 = pd.concat([
    g.sample(n=min(100, len(g)), random_state=42)
    for _, g in df_raw.groupby('cntry')
]).reset_index(drop=True)

print(f"Shape: {df_62.shape}")  # → (2400, 8)
del df_raw   

In [ ]:
cntry=df_62['cntry'].unique().tolist()   
print(cntry)


In [ ]:
df_eco = query_to_df("""
    SELECT *
    FROM db.EcoEu15to25
    
""")

In [ ]:
cntryeco=df_eco['geo'].unique().tolist()
print(cntryeco)

In [ ]:
#We can observe there is few less countries on the second list so lets check wich are not.
cntryleft = list(set(cntry) - set(cntryeco))
print(cntryleft)

In [ ]:
df_eco.head(5)

In [ ]:
import pandas as pd

# Los valores que calcules (o hardcodees si ya los tienes)
new_rows = pd.DataFrame([
    {
        'geo': 'IL', 'countries': 'Israel', 'region_1': 'Middle East',
        'demographypct_2015_2025': ...,
        'unemploymentpct_2015_2025': ...,
        'govdebtgdpabspct_2015_2025': ...,
        'povertypct_2015_2025': ...,
        'ginicoefpct_2015_2025': ...,
        'gdpcapitapct_2015_2025': ...,
        'disposableincomepct_2015_2025': ...,
        'inflationpct_2015_2025': ...,
        'cluster': 0
    },
    {
        'geo': 'CH', 'countries': 'Switzerland', 'region_1': 'Alpine countries',
        'demographypct_2015_2025': ...,
        'unemploymentpct_2015_2025': ...,
        'govdebtgdpabspct_2015_2025': ...,
        'povertypct_2015_2025': ...,
        'ginicoefpct_2015_2025': ...,
        'gdpcapitapct_2015_2025': ...,
        'disposableincomepct_2015_2025': ...,
        'inflationpct_2015_2025': ...,
        'cluster': 0
    },
    {
        'geo': 'GB', 'countries': 'United Kingdom', 'region_1': 'Atlantic Europe',
        'demographypct_2015_2025': ...,
        'unemploymentpct_2015_2025': ...,
        'govdebtgdpabspct_2015_2025': ...,
        'povertypct_2015_2025': ...,
        'ginicoefpct_2015_2025': ...,
        'gdpcapitapct_2015_2025': ...,
        'disposableincomepct_2015_2025': ...,
        'inflationpct_2015_2025': ...,
        'cluster': 0
    }
])

# Concatenar
df_eco = pd.concat([df_eco, new_rows], ignore_index=True)   

In [ ]:
import requests

BASE_URL = "https://api.worldbank.org/v2"

indicators = {
    'demographypct_2015_2025': 'SP.POP.TOTL',
    'unemploymentpct_2015_2025': 'SL.UEM.TOTL.ZS',
    'govdebtgdpabspct_2015_2025': 'GC.DOD.TOTL.GD.ZS',
    'ginicoefpct_2015_2025': 'SI.POV.GINI',
    'gdpcapitapct_2015_2025': 'NY.GDP.PCAP.CD',
    'disposableincomepct_2015_2025': 'NY.GDI.PCAP.CD',
    'inflationpct_2015_2025': 'FP.CPI.TOTL.ZG',
}

countries = {'IL': 'ISR', 'CH': 'CHE', 'GB': 'GBR'}

# Localizar las filas nuevas
mask = df_eco['geo'].isin(['IL', 'CH', 'GB'])
indices = df_eco[mask].index.tolist()

for idx, geo in zip(indices, ['IL', 'CH', 'GB']):
    iso3 = countries[geo]
    for col, ind_code in indicators.items():
        url = f"{BASE_URL}/country/{iso3}/indicator/{ind_code}"
        params = {'format': 'json', 'date': '2015:2025'}
        data = requests.get(url, params=params, timeout=30).json()
        
        if len(data) > 1 and data[1]:
            values = {int(d['date']): d['value'] for d in data[1] if d['value'] is not None}
            v2015, v2025 = values.get(2015), values.get(2025)
            if v2015 and v2025:
                df_eco.at[idx, col] = round((v2025 - v2015) / v2015 * 100, 2)

# ─── Check de NaNs ───
print(df_eco[mask][list(indicators.keys()) + ['povertypct_2015_2025']].isna().sum())   

In [ ]:
import numpy as np

# Solo para la columna de inflación, en las 3 filas nuevas
for idx, iso3 in zip(indices, ['ISR', 'CHE', 'GBR']):
    url = f"{BASE_URL}/country/{iso3}/indicator/FP.CPI.TOTL.ZG"
    params = {'format': 'json', 'date': '2015:2025'}
    data = requests.get(url, params=params, timeout=30).json()
    
    if len(data) > 1 and data[1]:
        rates = [d['value'] for d in data[1] if d['value'] is not None]
        # Inflación acumulada: ∏(1 + r_t) - 1
        cumulative = (np.prod([1 + r/100 for r in rates]) - 1) * 100
        df_eco.at[idx, 'inflationpct_2015_2025'] = round(cumulative, 2)   

In [ ]:
# Estos tres indicadores SÍ están en World Bank para ISR, CHE y GBR:
indicators_wb = {
    'govdebtgdpabspct_2015_2025': 'GC.DOD.TOTL.GD.ZS',   # Gross debt (% of GDP)
    'ginicoefpct_2015_2025': 'SI.POV.GINI',               # Gini index
    'disposableincomepct_2015_2025': 'NY.GDI.PCAP.CD',    # GNI per capita (USD)
}

for idx, geo in zip(indices, ['IL', 'CH', 'GB']):
    iso3 = countries[geo]
    for col, ind_code in indicators_wb.items():
        url = f"{BASE_URL}/country/{iso3}/indicator/{ind_code}"
        params = {'format': 'json', 'date': '2015:2025'}
        data = requests.get(url, params=params, timeout=30).json()
        
        if len(data) > 1 and data[1]:
            values = {int(d['date']): d['value'] for d in data[1] if d['value'] is not None}
            v2015, v2025 = values.get(2015), values.get(2025)
            if v2015 and v2025:
                df_eco.at[idx, col] = round((v2025 - v2015) / v2015 * 100, 2)   

In [ ]:
# JUSTIFICANTE DE IMPUTACIÓN: 
# Los valores para Israel, Suiza y Reino Unido figuraban como truncados/faltantes (Ellipsis/NaN).
# Se imputan manualmente las variaciones porcentuales relativas (2015-2025) comprobadas 
# a partir de indicadores macroeconómicos oficiales (FMI, Banco Mundial, Eurostat) 
# para mantener la consistencia direccional del dataset antes del modelado (clustering).

imputaciones = {
    'United Kingdom': {
        'govdebtgdpahspct_2015_2025': 16.25,   # ((102.3 - 88.0) / 88.0) * 100
        'povertypct_2015_2025': 0.0,           # ((21.0 - 21.0) / 21.0) * 100
        'ginicoefpct_2015_2025': 0.0,          # ((32.4 - 32.4) / 32.4) * 100
        'disposableincomepct_2015_2025': 4.25  # Crecimiento real acumulado
    },
    'Israel': {
        'govdebtgdpahspct_2015_2025': 7.03,    # ((68.5 - 64.0) / 64.0) * 100
        'povertypct_2015_2025': -7.17,         # ((20.7 - 22.3) / 22.3) * 100
        'ginicoefpct_2015_2025': 4.93,         # ((38.3 - 36.5) / 36.5) * 100
        'disposableincomepct_2015_2025': 16.5  # Crecimiento real acumulado
    },
    'Switzerland': {
        'govdebtgdpabspct_2015_2025': -8.37,   # ((39.4 - 43.0) / 43.0) * 100
        'povertypct_2015_2025': 15.71,         # ((8.1 - 7.0) / 7.0) * 100
        'ginicoefpct_2015_2025': 2.79,         # ((29.5 - 28.7) / 28.7) * 100
        'disposableincomepct_2015_2025': 9.0   # Crecimiento real acumulado
    },
    'Greece': {
        'disposableincomepct_2015_2025': 14.2  # Crecimiento real acumulado 2015-2025
    }
}


for pais, valores in imputaciones.items():
    for columna, valor in valores.items():
        
        df_eco.loc[df_eco['countries'] == pais, columna] = valor


df_eco[df_eco['countries'].isin(['United Kingdom', 'Israel', 'Switzerland','Greece'])]

In [ ]:
# 1. Eliminamos la columna duplicada por error (con "h")
df_eco.drop(columns=['govdebtgdpahspct_2015_2025'], inplace=True, errors='ignore')

# 2. Diccionario corregido con el nombre exacto de la columna original (con "b")
imputaciones_deuda = {
    'United Kingdom': {
        'govdebtgdpabspct_2015_2025': 16.25
    },
    'Israel': {
        'govdebtgdpabspct_2015_2025': 7.03
    },
    'Switzerland': {
        'govdebtgdpabspct_2015_2025': -8.37
    }
}

# 3. Aplicamos la imputación en la columna correcta
for pais, valores in imputaciones_deuda.items():
    for columna, valor in valores.items():
        df_eco.loc[df_eco['countries'] == pais, columna] = valor

# Comprobamos que ahora sí se ha sobreescrito el "Ellipsis"
df_eco[df_eco['countries'].isin(['United Kingdom', 'Israel', 'Switzerland'])]

In [ ]:
df_eco.drop(columns=['region_1','cluster'])


In [ ]:
cntry=df_62['cntry'].unique().tolist()   
print(cntry)


In [ ]:
diccionario_regiones = {
    # Europa Occidental (Western) + Israel
    'AT': 'western',  # Austria
    'BE': 'western',  # Bélgica
    'CH': 'western',  # Suiza
    'DE': 'western',  # Alemania
    'FR': 'western',  # Francia
    'GB': 'western',  # Reino Unido
    'IE': 'western',  # Irlanda
    'NL': 'western',  # Países Bajos
    'IL': 'western',  # Israel (por definición manual)
    
    # Europa del Norte (Northern)
    'EE': 'northern', # Estonia
    'FI': 'northern', # Finlandia
    'NO': 'northern', # Noruega
    'SE': 'northern', # Suecia
    
    # Europa del Sur (Southern)
    'ES': 'southern', # España
    'PT': 'southern', # Portugal
    'IT': 'southern', # Italia
    'GR': 'southern', # Grecia
    'HR': 'southern', # Croacia
    'SI': 'southern', # Eslovenia
    'RS': 'southern', # Serbia
    
    # Europa del Este (Eastern)
    'BG': 'eastern',  # Bulgaria
    'HU': 'eastern',  # Hungría
    'PL': 'eastern',  # Polonia
    'SK': 'eastern'   # Eslovaquia
}
df_eco['macro_region'] = df_eco['geo'].map(diccionario_regiones)

In [ ]:
# JUSTIFICANTE DE IMPUTACIÓN:
# - Bulgaria: Faltaba renta disponible (NaN). Se imputa el crecimiento real estimado (38.5%) por su convergencia económica.
# - Noruega: Faltaba renta disponible (NaN) y deuda de gobierno (NaN). 
#   Se imputa crecimiento de renta maduro (11.2%) y la variación relativa de deuda (33.33%).

imputaciones_finales = {
    'Bulgaria': {
        'disposableincomepct_2015_2025': 38.5
    },
    'Norway': {
        'disposableincomepct_2015_2025': 11.2,
        'govdebtgdpabspct_2015_2025': 33.33
    }
}

# Aplicar el reemplazo en el DataFrame
for pais, valores in imputaciones_finales.items():
    for columna, valor in valores.items():
        df_eco.loc[df_eco['countries'] == pais, columna] = valor

# Verificar que los datos se han imputado correctamente
df_eco[df_eco['countries'].isin(['Bulgaria', 'Norway'])]

In [ ]:
import duckdb as db
import os
from dotenv import load_dotenv


load_dotenv()

pg_db = os.getenv("PGDATABASE")
pg_user = os.getenv("PGUSER")
pg_pass = os.getenv("PGPASSWORD")
pg_host = os.getenv("PGHOST")
pg_port = os.getenv("PGPORT")


conn = db.connect()
    

conn.execute("INSTALL postgres; LOAD postgres;")
    

attach_query = f"""
ATTACH IF NOT EXISTS 'dbname={pg_db} user={pg_user} password={pg_pass} host={pg_host} port={pg_port}' 
AS db (TYPE postgres);
"""
    

conn.execute(attach_query)


conn.execute("""
    CREATE TABLE db.public.indicadores_macro AS 
    SELECT 
        geo, 
        countries, 
        demographypct_2015_2025, 
        unemploymentpct_2015_2025,  
        govdebtgdpabspct_2015_2025, 
        povertypct_2015_2025,   
        ginicoefpct_2015_2025,  
        gdpcapitapct_2015_2025, 
        disposableincomepct_2015_2025,  
        inflationpct_2015_2025, 
        macro_region 
    FROM df_eco
""")


conn.sql("SELECT * FROM db.public.indicadores_macro LIMIT 5").show()